# 11 Metrics Dashboard

Detailed, readable visual analysis for:
- next-event prediction quality (`MRR@10`, `Hit@10`)
- retention downstream quality (`ROC-AUC`, `PR-AUC`, `F1`, `LogLoss`) for `retention_7d` and `retention_14d`

This notebook uses default artifacts and explicitly keeps baseline models visible.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
PROJECT_ROOT

In [ ]:
seq_path = PROJECT_ROOT / "artifacts/reports/sequence_metrics.csv"
down_path = PROJECT_ROOT / "artifacts/reports/downstream_metrics.csv"

seq = pd.read_csv(seq_path)
down = pd.read_csv(down_path)

print("sequence rows:", len(seq), "| columns:", list(seq.columns))
print("downstream rows:", len(down), "| columns:", list(down.columns))
print("\nsequence models:", sorted(seq["model_name"].unique().tolist()))
print("downstream models:", sorted(down["model_name"].unique().tolist()))

# Human-readable model labels
LABELS = {
    "MostPopular": "MostPopular",
    "Markov1": "Markov1",
    "Transformer": "Transformer",
    "main_cls_logreg": "Main CLS + LogReg",
    "main_mean_logreg": "Main Mean + LogReg",
    "combined_cls_baseline_svd_logreg": "Combined (Main CLS + Baseline SVD)",
    "baseline_svd_logreg": "Baseline SVD + LogReg",
    "baseline_features_hgb_clean": "Baseline Features HGB (clean)",
    "baseline_features_logreg_clean": "Baseline Features LogReg (clean)",
    "baseline_features_hgb_legacy": "Baseline Features HGB (legacy)",
    "baseline_features_logreg_legacy": "Baseline Features LogReg (legacy)",
}

def pretty_model(name: str) -> str:
    return LABELS.get(name, name)

seq["model_pretty"] = seq["model_name"].map(pretty_model)
down["model_pretty"] = down["model_name"].map(pretty_model)

## 1) Next-Event Quality (`MRR@10`, `Hit@10`)

Focus on test split and compare Transformer to classic baselines.

In [ ]:
seq_test = seq[seq["split"].eq("test")].copy()
seq_test = seq_test.sort_values(["prefix_len", "model_name"])

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

sns.lineplot(
    data=seq_test,
    x="prefix_len",
    y="mrr_at_10",
    hue="model_pretty",
    marker="o",
    ax=axes[0],
)
axes[0].set_title("Test MRR@10 by Prefix Length")
axes[0].set_xlabel("Prefix Length")
axes[0].set_ylabel("MRR@10")
axes[0].legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")

sns.lineplot(
    data=seq_test,
    x="prefix_len",
    y="hit_at_10",
    hue="model_pretty",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Test Hit@10 by Prefix Length")
axes[1].set_xlabel("Prefix Length")
axes[1].set_ylabel("Hit@10")
axes[1].legend([], [], frameon=False)

plt.tight_layout()
plt.show()

seq_test[["model_pretty", "prefix_len", "mrr_at_10", "hit_at_10", "num_rows"]].sort_values(["model_pretty", "prefix_len"])

In [ ]:
pivot_mrr = seq_test.pivot_table(index="prefix_len", columns="model_pretty", values="mrr_at_10")
if "Transformer" in pivot_mrr.columns and "Markov1" in pivot_mrr.columns:
    lift_vs_markov = (pivot_mrr["Transformer"] - pivot_mrr["Markov1"]).rename("lift_mrr_vs_markov")
    print("Transformer lift over Markov1 (MRR@10):")
    display(lift_vs_markov.to_frame())

if "Transformer" in pivot_mrr.columns and "MostPopular" in pivot_mrr.columns:
    lift_vs_pop = (pivot_mrr["Transformer"] - pivot_mrr["MostPopular"]).rename("lift_mrr_vs_mostpopular")
    print("Transformer lift over MostPopular (MRR@10):")
    display(lift_vs_pop.to_frame())

## 2) Retention Downstream (7d / 14d)

Main plots for `ROC-AUC`, `PR-AUC`, `F1`, and `LogLoss` on test split.

By default we keep core models plus explicit baseline variants.

In [ ]:
preferred_models = [
    "baseline_features_hgb_clean",
    "baseline_features_hgb_legacy",
    "baseline_svd_logreg",
    "combined_cls_baseline_svd_logreg",
    "main_cls_logreg",
    "main_mean_logreg",
]
available = [m for m in preferred_models if m in set(down["model_name"])]
if not available:
    available = sorted(down["model_name"].unique().tolist())

down_core = down[down["model_name"].isin(available)].copy()
down_test = down_core[down_core["split"].eq("test")].copy()

print("Models in dashboard:")
for m in available:
    print("-", pretty_model(m))

down_test.groupby(["target", "model_pretty"])[["roc_auc", "pr_auc", "f1", "logloss"]].mean().sort_values(["target", "roc_auc"], ascending=[True, False])

In [ ]:
metrics = ["roc_auc", "pr_auc", "f1", "logloss"]
for target in sorted(down_test["target"].unique()):
    subset = down_test[down_test["target"].eq(target)].copy()
    fig, axes = plt.subplots(2, 2, figsize=(18, 11), sharex=True)
    axes = axes.ravel()
    for i, metric in enumerate(metrics):
        ax = axes[i]
        sns.lineplot(
            data=subset,
            x="prefix_len",
            y=metric,
            hue="model_pretty",
            marker="o",
            ax=ax,
        )
        ax.set_title(f"{target}: {metric.upper()} by Prefix Length (test)")
        ax.set_xlabel("Prefix Length")
        ax.set_ylabel(metric.upper())
        if i == 0:
            ax.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
        else:
            ax.legend([], [], frameon=False)
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmaps make model-vs-prefix comparison compact and quick to scan.
for metric in ["roc_auc", "pr_auc"]:
    for target in sorted(down_test["target"].unique()):
        hm = down_test[down_test["target"].eq(target)].pivot_table(
            index="model_pretty",
            columns="prefix_len",
            values=metric,
            aggfunc="mean",
        )
        plt.figure(figsize=(10, max(4, 0.6 * len(hm))))
        sns.heatmap(hm, annot=True, fmt=".3f", cmap="viridis")
        plt.title(f"{target}: {metric.upper()} heatmap (test)")
        plt.xlabel("Prefix Length")
        plt.ylabel("Model")
        plt.tight_layout()
        plt.show()

## 3) Validation-to-Test Stability

To avoid accidental over-reading test noise, inspect `valid - test` gap.
Smaller absolute gap usually means more stable generalization.

In [ ]:
vt = down_core[down_core["split"].isin(["valid", "test"])].copy()
agg = vt.pivot_table(
    index=["target", "prefix_len", "model_pretty"],
    columns="split",
    values=["roc_auc", "pr_auc"],
)
agg.columns = [f"{metric}_{split}" for metric, split in agg.columns]
agg = agg.reset_index()
agg["gap_roc_auc_valid_minus_test"] = agg["roc_auc_valid"] - agg["roc_auc_test"]
agg["gap_pr_auc_valid_minus_test"] = agg["pr_auc_valid"] - agg["pr_auc_test"]

display_cols = [
    "target", "prefix_len", "model_pretty",
    "roc_auc_valid", "roc_auc_test", "gap_roc_auc_valid_minus_test",
    "pr_auc_valid", "pr_auc_test", "gap_pr_auc_valid_minus_test",
]
agg_sorted = agg.sort_values(["target", "prefix_len", "gap_roc_auc_valid_minus_test"], ascending=[True, True, False])
agg_sorted[display_cols].head(40)

In [ ]:
# Winner per (target, prefix_len) chosen on valid ROC-AUC, then reported on test.
valid = down_core[down_core["split"].eq("valid")]
test = down_core[down_core["split"].eq("test")]

best_valid = (
    valid.sort_values(["target", "prefix_len", "roc_auc"], ascending=[True, True, False])
    .groupby(["target", "prefix_len"], as_index=False)
    .head(1)
)
valid_selected_test = test.merge(
    best_valid[["target", "prefix_len", "model_name"]],
    on=["target", "prefix_len", "model_name"],
    how="inner",
)
valid_selected_test = valid_selected_test.sort_values(["target", "prefix_len"])
valid_selected_test[["target", "prefix_len", "model_pretty", "roc_auc", "pr_auc", "f1", "logloss", "num_rows", "positive_rate"]]

## 4) Optional: Save Plots

Uncomment and run if you want all current figures saved as PNG files.

```python
out_dir = PROJECT_ROOT / "artifacts/reports/dashboard_plots"
out_dir.mkdir(parents=True, exist_ok=True)
for i, fig_num in enumerate(plt.get_fignums(), start=1):
    fig = plt.figure(fig_num)
    fig.savefig(out_dir / f"plot_{i:02d}.png", dpi=150, bbox_inches="tight")
print("Saved to", out_dir)
```